# Dự đoán sống sót

## Khởi tạo thí nghiệm

### Khai báo thư viện

In [1]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

### Tham số thực nghiệm

In [2]:
params_cfg = {
    # "action"   : "train_feat01",  
    # "feat_path": "../../exps/featbase_251028/data.npz",
    "seed"    : 42, # Set random seed
    "exp_dir" : os.path.abspath('../exps/output'),
    'exp_name': 'result_ensemble_learning',
    "data_dir": os.path.abspath("../exps/feature1"),
    "verbose" : True,
    "k_fold": 10,
}
params_cfg.update(**{
    "save_dir": os.path.abspath(f'{params_cfg["exp_dir"]}/{params_cfg["exp_name"]}')
})

for v in params_cfg:
    print(f'+ {v}: {params_cfg[v]}')

globals().update(**params_cfg)

+ seed: 42
+ exp_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output
+ exp_name: result_ensemble_learning
+ data_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\feature1
+ verbose: True
+ k_fold: 10
+ save_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_ensemble_learning


### Load dữ liệu đã tiền xử lí 

In [3]:
df_train = pd.read_excel(f'{params_cfg["data_dir"]}/train_df_preprocess.xlsx')
df_test = pd.read_excel(f'{params_cfg["data_dir"]}/test_df_preprocess.xlsx')

data = np.load(f'{params_cfg["data_dir"]}/feat_preprocess.npz', allow_pickle=True)

# Load feature columns
feat_cols = np.load(f'{params_cfg["data_dir"]}/feature_columns.npz', allow_pickle=True)
feature_columns = feat_cols['feature_columns']

# Chuyển về DataFrame
x= pd.DataFrame(data['x_train'], columns=feature_columns)
y = pd.Series(data['y_train'], name='Survived')
x_test_final = pd.DataFrame(data['x_test'], columns=feature_columns)

if params_cfg["verbose"]:
    print("-"*10, "information", "-"*10)
    print(f'train shape: {df_train.shape}')
    print(f'test shape: {df_test.shape}')
    print(f'train-col: {set(df_train.columns)}')
    print(f'test-col: {set(df_test.columns)}')
    print("Union:", set(df_train.columns).intersection(set(df_test.columns)))
    print("Difference:", set(df_train.columns).difference(set(df_test.columns)))

---------- information ----------
train shape: (891, 21)
test shape: (418, 20)
train-col: {'Ticket', 'Fare_Pclass', 'Parch', 'HasCabin', 'IsMother', 'SibSp', 'Name', 'Fare', 'TicketPrefix', 'Cabin', 'FamilySize', 'Survived', 'IsChild', 'Age', 'Pclass', 'PassengerId', 'Title', 'Embarked', 'Age*Pclass', 'Deck', 'Sex'}
test-col: {'Ticket', 'Fare_Pclass', 'Parch', 'HasCabin', 'IsMother', 'SibSp', 'Name', 'Fare', 'TicketPrefix', 'Cabin', 'FamilySize', 'IsChild', 'Age', 'Pclass', 'PassengerId', 'Title', 'Embarked', 'Age*Pclass', 'Deck', 'Sex'}
Union: {'Ticket', 'Fare_Pclass', 'Parch', 'HasCabin', 'IsMother', 'SibSp', 'Name', 'Fare', 'TicketPrefix', 'Cabin', 'FamilySize', 'IsChild', 'Age', 'Pclass', 'PassengerId', 'Title', 'Embarked', 'Age*Pclass', 'Deck', 'Sex'}
Difference: {'Survived'}


### Preprocess pipeline

In [4]:
# Preprocessor (ColumnTransformer)
num_features = [
    'Age','Fare','FamilySize','Fare_Pclass','Age*Pclass'
]
cat_features = [
    'Pclass','Sex','Embarked','Title','IsChild','IsMother',
    'Deck','HasCabin','TicketPrefix'
]

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features), ('cat', cat_transformer, cat_features)])

## Models Training

**Các mô hình dùng để phân tích**: `Logistic Regression`, `Random Forest`, `XGBoost`, `SVM`

Định nghĩa các pipeline mô hình học máy (dùng chung preprocessor, khác models)
- `Logistic Regression`: mô hình tuyến tính cơ bản (baseline)
- `Random Forest`: tập hợp nhiều cây quyết định, giúp giảm overfitting
- `XGBoost`: mô hình boosting mạnh mẽ, cho hiệu suất cao nhất
- `SVM`: bộ phân loại phi tuyến, phù hợp với ranh giới phức tạp

In [5]:
# --- Khởi tạo các pipeline ---
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000,random_state=42,C=1.0,solver='lbfgs'))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=120,max_depth=2,learning_rate=0.08,subsample=0.6,colsample_bytree=0.6,reg_lambda=3,reg_alpha=2, enable_categorical=False))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingClassifier(random_state=42,learning_rate=0.1))
    ])
}

In [6]:
# --- Khởi tạo các pipeline ---
weighted_models = {
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=120,max_depth=2,learning_rate=0.08,subsample=0.6,colsample_bytree=0.6,reg_lambda=3,reg_alpha=2, enable_categorical=False))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ]),
}

- Mỗi mô hình đều kết hợp với cùng bộ xử lý dữ liệu (preprocessor) để đảm bảo đầu vào nhất quán.
- Lưu trong dictionary models giúp dễ huấn luyện và so sánh kết quả giữa các thuật toán.

### Đánh giá từng model bằng cross-validation

#### Baseline

In [7]:
cv = StratifiedKFold(n_splits=params_cfg["k_fold"], shuffle=True, random_state=42)

results = {'Model': [], 'Metric': [], 'Score': []}

for name, model in models.items():
    acc_scores = cross_val_score(model, x, y, cv=cv, scoring='accuracy')
    f1_scores  = cross_val_score(model, x, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, x, y, cv=cv, scoring='roc_auc')

    # Lưu chi tiết từng lần CV
    for s in acc_scores:
        results['Model'].append(name)
        results['Metric'].append('Accuracy')
        results['Score'].append(s)
    for s in f1_scores:
        results['Model'].append(name)
        results['Metric'].append('F1 Score')
        results['Score'].append(s)
    for s in auc_scores:
        results['Model'].append(name)
        results['Metric'].append('ROC AUC')
        results['Score'].append(s)

    # Tính trung bình và độ lệch chuẩn
    acc_mean, acc_std = acc_scores.mean(), acc_scores.std()
    f1_mean, f1_std = f1_scores.mean(), f1_scores.std()
    auc_mean, auc_std = auc_scores.mean(), auc_scores.std()

    print(f"{name} CV Results:")
    print(f"  Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"  F1 Score: {f1_mean:.4f} ± {f1_std:.4f}")
    print(f"  ROC AUC:  {auc_mean:.4f} ± {auc_std:.4f}")
    print("-" * 40)

Logistic Regression CV Results:
  Accuracy: 0.8192 ± 0.0338
  F1 Score: 0.7568 ± 0.0479
  ROC AUC:  0.8750 ± 0.0377
----------------------------------------
Random Forest CV Results:
  Accuracy: 0.8271 ± 0.0321
  F1 Score: 0.7529 ± 0.0557
  ROC AUC:  0.8710 ± 0.0459
----------------------------------------
XGBoost CV Results:
  Accuracy: 0.8159 ± 0.0365
  F1 Score: 0.7477 ± 0.0587
  ROC AUC:  nan ± nan
----------------------------------------
SVM CV Results:
  Accuracy: 0.8350 ± 0.0227
  F1 Score: 0.7652 ± 0.0406
  ROC AUC:  0.8727 ± 0.0339
----------------------------------------
Gradient Boosting CV Results:
  Accuracy: 0.8282 ± 0.0291
  F1 Score: 0.7643 ± 0.0439
  ROC AUC:  0.8693 ± 0.0373
----------------------------------------


- Sử dụng StratifiedKFold (10-fold CV) để đánh giá 4 mô hình khác nhau.
- Tính 3 chỉ số: Accuracy, F1 Score, ROC AUC cho từng mô hình.
- Lưu và in kết quả trung bình ± độ lệch chuẩn để so sánh hiệu suất và độ ổn định giữa các mô hình.

#### Ensemble learning

In [8]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone

class WeightedEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """Custom ensemble với weighted average"""
    
    def __init__(self, models_dict, weights=(0.30, 0.45, 0.25)):
        self.models_dict = models_dict
        self.weights = weights
        self.svm = None
        self.xgb = None
        self.rf = None
    
    def fit(self, X, y):
        """Fit 3 models"""
        self.svm = clone(self.models_dict['SVM'])
        self.xgb = clone(self.models_dict['XGBoost'])
        self.rf = clone(self.models_dict['Random Forest'])
        
        self.svm.fit(X, y)
        self.xgb.fit(X, y)
        self.rf.fit(X, y)
        return self
    
    def predict_proba(self, X):
        """Weighted average of probabilities"""
        svm_proba = self.svm.predict_proba(X)
        xgb_proba = self.xgb.predict_proba(X)
        rf_proba = self.rf.predict_proba(X)
        
        # Weighted average
        w1, w2, w3 = self.weights
        final_proba = (w1 * svm_proba + w2 * xgb_proba + w3 * rf_proba)
        return final_proba
    
    def predict(self, X):
        """Predict class labels"""
        proba = self.predict_proba(X)
        return (proba[:, 1] > 0.5).astype(int)

In [ ]:
# === ENSEMBLE CONFIGURATIONS ===
from sklearn.ensemble import VotingClassifier


# 1. STACKING: SVM + XGBoost + Gradient Boosting
ensemble_1 = StackingClassifier(
    estimators=[
        ('svm', models['SVM']),
        ('xgb', models['XGBoost']),
        ('gb', models['Gradient Boosting'])
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=0.5),
    cv=5,
    n_jobs=-1
)

# 2. STACKING: Random Forest + XGBoost 
ensemble_2 = StackingClassifier(
    estimators=[
        ('rf', models['Random Forest']),
        ('xgb', models['XGBoost'])
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

# 3. VOTING SOFT: SVM + XGBoost + Random Forest 
ensemble_3 = VotingClassifier(
    estimators=[
        ('svm', models['SVM']),
        ('xgb', models['XGBoost']),
        ('rf', models['Random Forest'])
    ],
    voting='soft',
    weights=[1.2, 1.5, 1.0],  # XGBoost có trọng số cao hơn
    n_jobs=-1
)

ensemble_4 = WeightedEnsembleClassifier(
    models_dict=weighted_models,
    weights=(0.30, 0.45, 0.25)
)

In [10]:
# === EVALUATE ALL ENSEMBLES ===
ensemble_configs = {
    'Stack (SVM+XGB+GB)': ensemble_1,
    'Stack (RF+XGB)': ensemble_2,
    # 'Voting (SVM+XGB+RF)': ensemble_3,
    # 'Stack (All 5)': ensemble_4
    'Weighted Ensemble (SVM+XGB+RF)': ensemble_4
}

print("="*60)
print("ENSEMBLE MODEL COMPARISON")
print("="*60)

best_score = 0
best_name = ""

for name, ensemble in ensemble_configs.items():
    # Cross-validation
    acc = cross_val_score(ensemble, x, y, cv=cv, scoring='accuracy').mean()
    std_acc = acc.std()
    f1 = cross_val_score(ensemble, x, y, cv=cv, scoring='f1').mean()
    std_f1 = f1.std()
    auc = cross_val_score(ensemble, x, y, cv=cv, scoring='roc_auc').mean()
    std_auc = auc.std()
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f} (+/- {std_acc:.4f})")
    print(f"  F1 Score: {f1:.4f} (+/- {std_f1:.4f})")
    print(f"  AUC: {auc:.4f} (+/- {std_auc:.4f})")

    # Track best
    if acc > best_score:
        best_score = acc
        best_name = name

print("\n" + "="*60)
print(f"🏆 BEST ENSEMBLE: {best_name} ({best_score:.4f})")
print("="*60)

ENSEMBLE MODEL COMPARISON

Stack (SVM+XGB+GB):
  Accuracy: 0.8316 (+/- 0.0000)
  F1 Score: 0.7681 (+/- 0.0000)
  AUC: 0.8753 (+/- 0.0000)

Stack (RF+XGB):
  Accuracy: 0.8260 (+/- 0.0000)
  F1 Score: 0.7625 (+/- 0.0000)
  AUC: 0.8722 (+/- 0.0000)

Weighted Ensemble (SVM+XGB+RF):
  Accuracy: 0.8282 (+/- 0.0000)
  F1 Score: 0.7593 (+/- 0.0000)
  AUC: nan (+/- nan)

🏆 BEST ENSEMBLE: Stack (SVM+XGB+GB) (0.8316)


## Xuất ra file kết quả

In [ ]:
# === XUẤT KẾT QUẢ CHO CÁC ENSEMBLE MODELS ===

# Tạo thư mục lưu kết quả
save_dir = params_cfg["save_dir"]
os.makedirs(save_dir, exist_ok=True)

ensemble_submissions = {}

print("="*60)
print("TRAINING AND GENERATING SUBMISSIONS FOR ENSEMBLE MODELS")
print("="*60)

# Dictionary chứa các ensemble models
ensemble_models = {
    'Stack_SVM_XGB_GB': ensemble_1,
    'Stack_RF_XGB': ensemble_2,
    # 'Voting_SVM_XGB_RF': ensemble_3,
    'Weighted_Ensemble_SVM_XGB_RF': ensemble_4
}

for name, ensemble_model in ensemble_models.items():
    print(f"\nTraining {name}...")
    # Dự đoán trên test set
    predictions = ensemble_model.predict(x_test_final)
    
    # Lưu vào dictionary
    filename = f'submission_ensemble_{name.lower()}.csv'
    ensemble_submissions[filename] = predictions
    
    print(f"✓ {name} training completed!")
    print(f"  Predictions shape: {predictions.shape}")
    print(f"  Survived count: {predictions.sum()}, Not survived: {len(predictions) - predictions.sum()}")
    print("-" * 60)

# Lưu tất cả ensemble submissions ra file CSV
print("\n📁 Saving ensemble submission files...")
for filename, predictions in ensemble_submissions.items():
    submission = pd.DataFrame({
        'PassengerId': df_test['PassengerId'],
        'Survived': predictions.astype(int)
    })
    
    filepath = os.path.join(save_dir, filename)
    submission.to_csv(filepath, index=False)
    print(f"✓ Saved: {filepath}")

print(f"\n🎉 Created {len(ensemble_submissions)} ensemble submission files in:")
print(f"   {save_dir}")

# Tổng hợp thông tin
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total individual models: {len(models)}")
print(f"Total ensemble models: {len(ensemble_models)}")
print(f"Total submission files: {len(ensemble_submissions)}")
print("="*60)

TRAINING AND GENERATING SUBMISSIONS FOR ENSEMBLE MODELS

Training Stack_SVM_XGB_GB...
✓ Stack_SVM_XGB_GB training completed!
  Predictions shape: (418,)
  Survived count: 152, Not survived: 266
------------------------------------------------------------

Training Stack_RF_XGB...
✓ Stack_RF_XGB training completed!
  Predictions shape: (418,)
  Survived count: 164, Not survived: 254
------------------------------------------------------------

Training Weighted_Ensemble_SVM_XGB_RF...
✓ Weighted_Ensemble_SVM_XGB_RF training completed!
  Predictions shape: (418,)
  Survived count: 154, Not survived: 264
------------------------------------------------------------

📁 Saving ensemble submission files...
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_ensemble_learning\submission_ensemble_stack_svm_xgb_gb.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_ensemble_learning\submission_ensemble_stack_rf_xgb.csv
✓ Saved: d:\ML_git\Machine_L